# 7 — Radial distribution function (2D lateral, P–P)

All-atom CHARMM36 binary bilayer (DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA).

In-plane g(r) between phosphates for every lipid pair. A single frame gives a noisy g(r); a trajectory smooths it. (Note: needs enough lipids per leaflet.)


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy scipy matplotlib
%matplotlib inline
import os, re, itertools
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config — edit these

You have **no trajectory**, so `TRAJ = None` loads the single `.gro`
frame (a **snapshot**). When you have an `.xtc`/`.trr`, set `TRAJ` to it
and every cell below averages over frames automatically.

In [ ]:
TOPOLOGY = "md.tpr"        # topology (has bonds + masses)
STRUCTURE= "md.gro"        # single-frame structure (.gro or .pdb)
TRAJ     = None            # e.g. "md_center.xtc" when you have one; None -> use STRUCTURE

LIPIDS   = ["DOPC", "DOPA"]        # the TWO lipids in THIS system
HEAD     = "P"                     # headgroup / leaflet marker
STEP     = 1                       # frame stride (only matters with a trajectory)
OUTDIR   = "results"; os.makedirs(OUTDIR, exist_ok=True)

## Load

In [ ]:
u = mda.Universe(TOPOLOGY, TRAJ) if TRAJ else mda.Universe(TOPOLOGY, STRUCTURE)
has_bonds = len(getattr(u, "bonds", [])) > 0
present   = [r for r in LIPIDS if len(u.select_atoms(f"resname {r}")) > 0]
nframes   = len(u.trajectory)
print(f"{len(u.atoms)} atoms, {nframes} frame(s), bonds={has_bonds}")
print("Lipids present:", present)
if nframes == 1:
    print("NOTE: single frame -> snapshot only (no time averaging).")

## 2D lateral RDF

In [ ]:
RMAX_nm=3.0; NBINS=100; rmax=RMAX_nm*10
edges=np.linspace(0,rmax,NBINS+1); ctr=0.5*(edges[1:]+edges[:-1]); shell=np.pi*(edges[1:]**2-edges[:-1]**2)
def pdist2d(a,b,Lx,Ly,same):
    dx=a[:,None,0]-b[None,:,0]; dy=a[:,None,1]-b[None,:,1]
    dx-=Lx*np.round(dx/Lx); dy-=Ly*np.round(dy/Ly); r=np.sqrt(dx*dx+dy*dy)
    return r[np.triu_indices(r.shape[0],1)] if same else r.ravel()
sel={r:u.select_atoms(f"name {HEAD} and resname {r}") for r in present}
pairs=list(itertools.combinations_with_replacement(present,2))
hist={p:np.zeros(NBINS) for p in pairs}; norm={p:0.0 for p in pairs}
for ts in u.trajectory[::STEP]:
    Lx,Ly=ts.dimensions[:2]; area=Lx*Ly; xy={r:sel[r].positions[:,:2] for r in present}
    for A,B in pairs:
        same=A==B
        hist[(A,B)]+=np.histogram(pdist2d(xy[A],xy[B],Lx,Ly,same),bins=edges)[0]
        Na,Nb=len(xy[A]),len(xy[B]); norm[(A,B)]+=0.5*Na*((Nb-1)/area) if same else Na*(Nb/area)
plt.figure(figsize=(7,4.5))
for A,B in pairs:
    if norm[(A,B)]==0: continue
    g=hist[(A,B)]/(norm[(A,B)]*shell)
    np.savetxt(f"{OUTDIR}/rdf_{A}-{B}.dat",np.column_stack([ctr/10,g]),header="r_nm g")
    plt.plot(ctr/10,g,label=f"{A}-{B}")
plt.axhline(1,color="k",lw=.6,ls="--"); plt.xlabel("r (nm)"); plt.ylabel("g(r)")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title("2D lateral RDF (P-P)"); plt.show()